# 01 — Telemetry Exploration: Monaco GP 2026

Goal of this notebook:
1. Load the 2026 Monaco Grand Prix race session via FastF1
2. Understand the structure of a `Session` and a `Lap` object
3. Identify a fast, clean lap suitable for telemetry analysis
4. Plot the basic telemetry channels and identify braking events
5. Build initial intuition for what 2026 brake telemetry actually looks like

Monaco is chosen because it has the densest braking-event profile of any 2026 race — every lap is a chain of heavy braking zones, which is exactly what we want for a brake-focused project.

In [4]:
import fastf1
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Make plots a bit nicer by default
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.dpi'] = 100

print(f"FastF1 version: {fastf1.__version__}")
print(f"Pandas version: {pd.__version__}")

#when something breaks later, version mismatches are one of the first things to check. 

FastF1 version: 3.8.3
Pandas version: 2.3.3


In [5]:
# Enable the FastF1 cache so we don't re-download every time
# Path is relative to where the notebook is run from
fastf1.Cache.enable_cache('../data/raw')

#FastF1 pulls data from the official F1 timing API and from open-source archives. The first time you load a session it downloads everything; subsequent loads read from disk. The cache will live in data/raw/ which is gitignore'd, so it won't bloat the repo.

In [6]:
# Load 2026 Monaco GP QUALIFYING
# 'Q' = Qualifying session
session = fastf1.get_session(2026, 'Monaco', 'Q')

session.load(laps=True, telemetry=True, weather=True, messages=False)

print(f"Loaded: {session.event['EventName']} - {session.name}")
print(f"Date: {session.date}")
print(f"Total laps in session data: {len(session.laps)}")

#What session.load() does: pulls down the lap timings, per-lap telemetry, weather, and skips race control messages (we don't need them yet). 

core           INFO 	Loading data for Monaco Grand Prix - Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	

Loaded: Monaco Grand Prix - Qualifying
Date: 2026-06-06 14:00:00
Total laps in session data: 484


In [7]:
# Top of the results table — who finished where
print("=== Race Result (Top 10) ===")
print(session.results[['Position', 'Abbreviation', 'TeamName', 'Time', 'Status']].head(10))

print("\n=== Weather Summary ===")
weather = session.weather_data
print(f"Air temp:     {weather['AirTemp'].mean():.1f} °C  (min {weather['AirTemp'].min():.1f}, max {weather['AirTemp'].max():.1f})")
print(f"Track temp:   {weather['TrackTemp'].mean():.1f} °C  (min {weather['TrackTemp'].min():.1f}, max {weather['TrackTemp'].max():.1f})")
print(f"Humidity:     {weather['Humidity'].mean():.1f} %")
print(f"Rain flag:    {weather['Rainfall'].any()}")

#before any analysis, you sanity-check what you're working with. Was it wet? Were temperatures stable? Did the race go full distance? If we see Rain flag: True, our entire braking analysis just got a lot messier — we'd want to either pick a different session or restrict to dry laps.

=== Race Result (Top 10) ===
    Position Abbreviation         TeamName Time Status
12       1.0          ANT         Mercedes  NaT       
3        2.0          VER  Red Bull Racing  NaT       
44       3.0          HAM          Ferrari  NaT       
16       4.0          LEC          Ferrari  NaT       
6        5.0          HAD  Red Bull Racing  NaT       
63       6.0          RUS         Mercedes  NaT       
81       7.0          PIA          McLaren  NaT       
1        8.0          NOR          McLaren  NaT       
10       9.0          GAS           Alpine  NaT       
30      10.0          LAW     Racing Bulls  NaT       

=== Weather Summary ===
Air temp:     23.7 °C  (min 23.0, max 24.5)
Track temp:   38.5 °C  (min 32.3, max 47.4)
Humidity:     54.9 %
Rain flag:    False


In [8]:
# In qualifying, we don't use pick_quicklaps() — that filter is for filtering out
# slow race laps (in-laps, out-laps, SC laps). In qualifying every lap is either
# a flying lap, an out-lap, or an in-lap, and the lap times themselves cleanly
# distinguish them.

# Just pick the single fastest lap of the session — this is the pole position lap
fastest = session.laps.pick_fastest()

print(f"Pole position lap:")
print(f"  Driver:        {fastest['Driver']} ({fastest['Team']})")
print(f"  Lap number:    {int(fastest['LapNumber'])}")
print(f"  Lap time:      {fastest['LapTime']}")
print(f"  Compound:      {fastest['Compound']}")
print(f"  Tyre age:      {int(fastest['TyreLife'])} laps")
print(f"  Sector 1:      {fastest['Sector1Time']}")
print(f"  Sector 2:      {fastest['Sector2Time']}")
print(f"  Sector 3:      {fastest['Sector3Time']}")

Pole position lap:
  Driver:        ANT (Mercedes)
  Lap number:    27
  Lap time:      0 days 00:01:12.051000
  Compound:      SOFT
  Tyre age:      3 laps
  Sector 1:      0 days 00:00:18.934000
  Sector 2:      0 days 00:00:33.989000
  Sector 3:      0 days 00:00:19.128000


In [ ]:
# get_telemetry() returns a DataFrame combining car data (speed, throttle, brake, etc.)
# with position data (X, Y, Z) and adds derived columns like Distance and
# RelativeDistance (position along the lap, very useful for plotting).
tel = fastest.get_telemetry()

print(f"Telemetry shape: {tel.shape}  (rows × columns)")
print(f"Telemetry columns:\n{list(tel.columns)}")
print(f"\nFirst 3 rows:")
tel.head(3)

#each row is one sample in time (FastF1 typically interpolates to a uniform grid; expect ~5–10ms per sample, so a 72-second lap gives several thousand rows). The columns are the channels we listed in the scope doc.

In [ ]:
# Check the channels we care about
channels = ['Speed', 'Throttle', 'Brake', 'nGear', 'RPM', 'Distance']

for col in channels:
    if col in tel.columns:
        s = tel[col]
        # Brake is boolean, others are numeric
        if s.dtype == bool:
            print(f"{col:12s} | dtype={str(s.dtype):8s} | True samples: {s.sum()}/{len(s)} ({s.mean()*100:.1f}%)")
        else:
            print(f"{col:12s} | dtype={str(s.dtype):8s} | min={s.min():>8.2f}  max={s.max():>8.2f}  mean={s.mean():>8.2f}")
    else:
        print(f"{col:12s} | NOT PRESENT")

print(f"\nLap distance covered: {tel['Distance'].iloc[-1]:.0f} m")
print(f"Number of samples on this lap: {len(tel)}")
print(f"Effective sample rate: {len(tel) / fastest['LapTime'].total_seconds():.1f} Hz")

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)

# Speed over distance
axes[0].plot(tel['Distance'], tel['Speed'], color='steelblue', linewidth=1.2)
axes[0].set_ylabel('Speed (km/h)')
axes[0].set_title(f"Monaco GP 2026 Q — Pole lap by {fastest['Driver']} ({fastest['LapTime']})")
axes[0].grid(True, alpha=0.3)

# Throttle as a percentage
axes[1].plot(tel['Distance'], tel['Throttle'], color='seagreen', linewidth=1.2)
axes[1].set_ylabel('Throttle (%)')
axes[1].set_ylim(-5, 105)
axes[1].grid(True, alpha=0.3)

# Brake as shaded regions (since it's boolean, a line plot is ugly)
# We'll fill in red where the driver is braking
axes[2].fill_between(tel['Distance'], 0, tel['Brake'].astype(int),
                     color='crimson', alpha=0.6, step='post')
axes[2].set_ylabel('Brake (on/off)')
axes[2].set_ylim(-0.1, 1.1)
axes[2].set_yticks([0, 1])
axes[2].set_xlabel('Distance along lap (m)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# A "braking event" is a contiguous stretch where Brake == True.
# To find them, we look at where Brake transitions from False -> True (start)
# and True -> False (end).

brake = tel['Brake'].astype(int).values  # 0/1 array
# np.diff gives us +1 at the start of each braking event, -1 at the end
edges = np.diff(brake, prepend=0, append=0)
starts = np.where(edges == 1)[0]
ends = np.where(edges == -1)[0] - 1  # -1 because diff shifts by one

# Sanity: number of starts should equal number of ends
assert len(starts) == len(ends), "Mismatched start/end counts — something is off"

print(f"Number of braking events detected: {len(starts)}")

# Build a small table summarizing each event
events = []
for i, (s_idx, e_idx) in enumerate(zip(starts, ends)):
    d_start = tel['Distance'].iloc[s_idx]
    d_end = tel['Distance'].iloc[e_idx]
    v_start = tel['Speed'].iloc[s_idx]
    v_end = tel['Speed'].iloc[e_idx]
    t_start = tel['Time'].iloc[s_idx].total_seconds()
    t_end = tel['Time'].iloc[e_idx].total_seconds()
    duration = t_end - t_start

    events.append({
        'event': i + 1,
        'dist_start_m': round(d_start, 0),
        'dist_end_m': round(d_end, 0),
        'duration_s': round(duration, 3),
        'v_start_kmh': round(v_start, 1),
        'v_end_kmh': round(v_end, 1),
        'delta_v_kmh': round(v_start - v_end, 1),
    })

events_df = pd.DataFrame(events)
print("\nBraking events on this lap:")
events_df

#np.diff computes the difference between each pair of consecutive samples. If brake goes from 0 to 1, the diff is +1 (event starts). If it goes from 1 to 0, the diff is -1 (event ends). The prepend=0, append=0 is a trick to make sure we catch events that start at the very first sample or end at the very last one.
#Then for each event we record where it happens on track, how long it lasted, and what the speed change was. The speed change (delta_v_kmh) is your immediate proxy for how heavy the braking was — a 200 km/h speed loss is a much bigger event than a 30 km/h brush.